# 01 — Hugging Face VLM Inference

Learn the standard processor → model → generation pipeline used by modern open-weight vision-language models.

> **Course principle:** every notebook separates **general-purpose knowledge** from **Manga Task application**.

## The general inference pattern

A VLM normally receives image(s) plus text instructions. The processor turns both into tensors. The model then generates text autoregressively.

Key objects to understand: processor, tokenizer, pixel values, chat template, generation config, decoded output.

In [1]:
from transformers import AutoProcessor
from transformers import Qwen2_5_VLForConditionalGeneration, AutoTokenizer, AutoProcessor
from qwen_vl_utils import process_vision_info
# from qwen_vl_utils import process_vision_info
# Replace with a compatible open-weight VLM checkpoint you choose.
# MODEL_ID = "YOUR_OPEN_WEIGHT_VLM"

MODEL_PATH = r"C:\models\HF\Qwen2.5-VL-3B-Instruct"

processor = AutoProcessor.from_pretrained(
    MODEL_PATH,
    local_files_only=True
)

model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
    MODEL_PATH,
    local_files_only=True,
    torch_dtype="auto",
    device_map="auto"
)


Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

Some parameters are on the meta device because they were offloaded to the disk and cpu.


In [12]:
img_path = r"istockphoto-470089276-612x612.jpg"
messages = [
    {
        "role": "user",
        "content": [
            {
                "type": "image",
                "image": img_path,
            },
            {"type": "text", "text": "Describe this image."},
        ],
    }
]

# print(processor.chat_template)
prompt = processor.apply_chat_template(
    messages,
    tokenize = False,
    add_generation_prompt = True
)

print(prompt)
image_inputs, vid_inputs = process_vision_info(messages)

inputs = processor(
    text=[prompt],
    images=image_inputs,
    padding=True,
    return_tensors="pt",
)
inputs = inputs.to("cuda")
print(image_inputs)
for key in dict(inputs).keys() :
    print(key, dict(inputs)[key].shape)

<|im_start|>system
You are a helpful assistant.<|im_end|>
<|im_start|>user
<|vision_start|><|image_pad|><|vision_end|>Describe this image.<|im_end|>
<|im_start|>assistant

[<PIL.Image.Image image mode=RGB size=616x420 at 0x1A5DEB6C1D0>]
input_ids torch.Size([1, 355])
attention_mask torch.Size([1, 355])
mm_token_type_ids torch.Size([1, 355])
pixel_values torch.Size([1320, 1176])
image_grid_thw torch.Size([1, 3])


In [ ]:
generated_ids = model.generate(**inputs, max_new_tokens=128)
generated_ids_trimmed = [
    # out_ids[:] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)              # ------ FOR FULL MESSAGE + OUTPUT
    out_ids[len(in_ids):] for in_ids, out_ids in zip(inputs.input_ids, generated_ids)     # ------ FOR just OUTPUT
]
output_text = processor.batch_decode(
    generated_ids_trimmed, skip_special_tokens=True, clean_up_tokenization_spaces=False
)
print(output_text)

["system\nYou are a helpful assistant.\nuser\nDescribe this image.\nassistant\nThe image depicts a serene and cozy scene featuring a light-colored cat lying on a bed. The cat appears to be in a relaxed state, with its body stretched out and its head resting on the bed's surface. The cat's fur is soft and fluffy, contributing to the overall calm and peaceful atmosphere of the image.\n\nThe bed itself is covered with a light-colored blanket or duvet that matches the cat's fur color. The bedding looks soft and comfortable, suggesting that the environment is warm and inviting. The background includes a curtain or drapery that adds to the homely feel of the setting. The curtain is also light-colored,"]


In [ ]:
generation_kwargs = {
    "max_new_tokens": 512,
    "do_sample": False,
}
generation_kwargs

## Manga-task prompting

Your prompt should explicitly define inclusion/exclusion rules, reading order, and the output schema. A single page baseline is useful before attempting three-page consistency.

In [ ]:
MANGA_PROMPT = '''
Read this English manga page.

Return only story text that should count:
- dialogue
- thoughts
- narration
- clear vocalisations
- punctuation-only speech balloons

Exclude:
- sound effects
- signs/clothing/object text
- credits/logos/page numbers
- editor/scanlator notes
- text inside phones, letters, documents, interfaces

Preserve visible punctuation and repetitions.
Return JSON only.
'''
print(MANGA_PROMPT)